#Phase 3

In [ ]:
# ── CELL 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 2: Prepare Kermany Dataset ──────────────────────────────────────────
# The chest_xray dataset should already be on your Drive from Assignment 2.
# Update the zip_path below if needed, then run this cell.

import zipfile, os

# *** Please update this path to where your chest_xray.zip is located ***
zip_path     = '/content/drive/MyDrive/archive.zip'   # Updated to a more generic path, adjust if different
extract_path = '/content/archive/'

if not os.path.exists(extract_path + 'chest_xray'):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_path)
    print("Dataset unzipped.")
else:
    print("Dataset already present.")

In [ ]:
# ── CELL 3: Install Dependencies ─────────────────────────────────────────────
# medmnist is new; rest is same as A2
!pip install -q transformers accelerate medmnist

In [ ]:
# ── CELL 4: Imports & Reproducibility ────────────────────────────────────────
import copy, random, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset, Dataset
from torchvision import transforms, datasets
from transformers import ViTForImageClassification, ViTImageProcessor
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve
)

from medmnist import BloodMNIST, INFO as MEDMNIST_INFO

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

In [ ]:
# ── CELL 5: Configuration ─────────────────────────────────────────────────────

# Paths
KERMANY_DIR = "/content/archive/chest_xray"
SAVE_DIR    = "/content/drive/MyDrive/DL_Assignment3/results/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Training hyperparameters (same as A2 where applicable)
BATCH_SIZE   = 16     # Reduced from 32 to prevent CUDA out of memory
MAX_EPOCHS   = 12     # reduced from 30 — layer freezing converges faster
PATIENCE     = 4      # early stopping on val F1
LR           = 1e-4
WEIGHT_DECAY = 1e-4
LR_FACTOR    = 0.5
LR_PATIENCE  = 3

# ── Assignment 3 NEW parameters (the improvements) ────────────────────────────
FREEZE_N     = 10     # freeze first 10 of 12 ViT encoder blocks
LABEL_SMOOTH = 0.1    # label smoothing epsilon

# ── Assignment 2 reported test results (for comparison table) ─────────────────
# These are the actual numbers from our A2 results_comparison.csv
A2_RESULTS = {
    'Accuracy (%)':    91.67,
    'Recall (%)':      92.56,
    'Specificity (%)': 90.17,
    'F1 (%)':          93.28,
    'AUC':             0.97,
}

print("Assignment 3 Configuration:")
print(f"  Freeze first {FREEZE_N}/12 ViT encoder blocks  → ~14M trainable params (vs ~86M in A2)")
print(f"  Label smoothing ε = {LABEL_SMOOTH}")
print(f"  Max epochs = {MAX_EPOCHS}, early-stop patience = {PATIENCE}")
print(f"  A2 baseline test accuracy: {A2_RESULTS['Accuracy (%)']}%")

In [ ]:
# ── CELL 6: Kermany Data Loading (fixed val transform bug from A2) ────────────
#
# BUG IN A2: val_ds was created with random_split which inherited train_transform
#            (augmentation). val_ds_eval was created from train_ds indices (training
#            data!), not val indices, and was never actually used in val_loader.
#
# FIX: Load full_train_dir TWICE — once with train_tfm (for training Subset),
#      once with eval_tfm (for validation Subset). Share the same index split.
#      This guarantees val images are NEVER augmented.

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

# Same transforms as A2
train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])
eval_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

# Load twice: same images, different transforms
full_aug  = datasets.ImageFolder(f"{KERMANY_DIR}/train", transform=train_tfm)
full_eval = datasets.ImageFolder(f"{KERMANY_DIR}/train", transform=eval_tfm)
test_ds_k = datasets.ImageFolder(f"{KERMANY_DIR}/test",  transform=eval_tfm)
CLASS_NAMES_K = full_aug.classes   # ['NORMAL', 'PNEUMONIA']

# Reproducible 80/20 index split
N = len(full_aug)
rng_split = np.random.default_rng(SEED)
all_idx   = rng_split.permutation(N)
split_pt  = int(0.8 * N)
train_idx = all_idx[:split_pt].tolist()
val_idx   = all_idx[split_pt:].tolist()

train_ds_k = Subset(full_aug,  train_idx)   # augmented — for training
val_ds_k   = Subset(full_eval, val_idx)     # clean (eval_tfm) — for validation (FIXED)

# Class-weighted sampler (same logic as A2)
train_labels_k = [full_aug.targets[i] for i in train_idx]
counts_k = np.bincount(train_labels_k)
cw_k = torch.tensor(1.0 / counts_k, dtype=torch.float)
cw_k = (cw_k / cw_k.sum() * 2).to(device)

sample_w_k = torch.tensor([cw_k[l].item() for l in train_labels_k])
sampler_k  = WeightedRandomSampler(sample_w_k, len(sample_w_k), replacement=True)

train_loader_k = DataLoader(train_ds_k, batch_size=BATCH_SIZE, sampler=sampler_k,
                             num_workers=2, pin_memory=True)
val_loader_k   = DataLoader(val_ds_k,   batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
test_loader_k  = DataLoader(test_ds_k,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f"Kermany  — Train: {len(train_ds_k)} | Val: {len(val_ds_k)} | Test: {len(test_ds_k)}")
print(f"Classes  : {CLASS_NAMES_K}")
print(f"Counts   — NORMAL: {counts_k[0]}, PNEUMONIA: {counts_k[1]}")
print(f"CW       — NORMAL: {cw_k[0]:.4f}, PNEUMONIA: {cw_k[1]:.4f}")

In [ ]:
# ── CELL 7: BloodMNIST Data Loading (Cross-Domain Dataset) ───────────────────
#
# BloodMNIST (medmnist package):
#   - Task: 8-class blood cell type classification
#   - Size: 11,959 train | 1,712 val | 3,421 test
#   - Images: 28×28 RGB (resized to 224×224 for ViT)
#   - Domain: hematology microscopy (different from chest radiology)
#   - Auto-downloads ~41 MB, no Kaggle account needed
#
# WHY THIS DATASET:
#   - No overlap with Kermany (different modality, different organ system)
#   - 8 classes → harder than binary Kermany → realistic, non-trivial results
#   - Well-established benchmark with known baselines
#   - Tests whether our ViT training approach generalises across medical domains

blood_train_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])
blood_eval_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

# Download BloodMNIST (first run downloads ~41 MB to ~/.medmnist/)
_b_train = BloodMNIST(split='train', transform=blood_train_tfm, download=True)
_b_val   = BloodMNIST(split='val',   transform=blood_eval_tfm,  download=True)
_b_test  = BloodMNIST(split='test',  transform=blood_eval_tfm,  download=True)

# MedMNIST returns labels as numpy arrays of shape (1,) — wrap to return ints
class MedMNISTWrapper(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        img, label = self.ds[i]
        return img, int(label[0])   # label shape (1,) → scalar int

train_ds_b = MedMNISTWrapper(_b_train)
val_ds_b   = MedMNISTWrapper(_b_val)
test_ds_b  = MedMNISTWrapper(_b_test)

# Class info from MedMNIST metadata
NUM_BLOOD_CLASSES = len(MEDMNIST_INFO['bloodmnist']['label'])
CLASS_NAMES_B = list(MEDMNIST_INFO['bloodmnist']['label'].values())

train_loader_b = DataLoader(train_ds_b, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=2, pin_memory=True)
val_loader_b   = DataLoader(val_ds_b,   batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
test_loader_b  = DataLoader(test_ds_b,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f"BloodMNIST — Train: {len(train_ds_b)} | Val: {len(val_ds_b)} | Test: {len(test_ds_b)}")
print(f"Classes ({NUM_BLOOD_CLASSES}): {CLASS_NAMES_B}")

In [ ]:
# ── CELL 8: Model Builder (with Layer Freezing) ───────────────────────────────

def build_vit(num_labels, freeze_n=FREEZE_N):
    """
    Load pretrained ViT-Base-Patch16-224 and apply layer freezing.
    The patch embeddings and first `freeze_n` encoder blocks are frozen.
    Only the last (12 - freeze_n) blocks + final LayerNorm + classifier head
    remain trainable.

    MOTIVATION:
      In A2, all 86M parameters were updated, causing the model to memorise
      the small Kermany training set (train loss → 0, train F1 → 1) without
      proportional improvement in test accuracy (91.67%).  Freezing early
      layers preserves general visual features learned from ImageNet
      pretraining while only adapting the task-specific top layers.
    """
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224",
        num_labels=num_labels,
        ignore_mismatched_sizes=True,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    )

    # Freeze patch + positional embeddings
    for p in model.vit.embeddings.parameters():
        p.requires_grad = False

    # Freeze first freeze_n encoder blocks (0-indexed)
    for i, block in enumerate(model.vit.encoder.layer):
        if i < freeze_n:
            for p in block.parameters():
                p.requires_grad = False

    # LayerNorm after encoder + classifier head remain trainable
    n_total     = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100.0 * n_trainable / n_total

    print(f"  ViT-Base built  |  {num_labels} output class(es)")
    print(f"  Frozen: embeddings + encoder blocks 0-{freeze_n-1} of 12")
    print(f"  Trainable: {n_trainable:,} / {n_total:,} params ({pct:.1f}%)")

    return model.to(device)


print("build_vit() defined — call it in the experiment cells.")

In [ ]:
# ── CELL 9: Training & Evaluation Helper Functions ────────────────────────────

def train_epoch(model, loader, criterion, optimizer):
    """One training epoch. Returns average loss."""
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, leave=False, desc='  [train]'):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs).logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate model. Returns dict with loss, preds, labels, probs."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in tqdm(loader, leave=False, desc='  [eval]'):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs).logits
        total_loss += criterion(logits, labels).item()
        probs = torch.softmax(logits, dim=1)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return {
        'loss':   total_loss / len(loader),
        'preds':  np.array(all_preds),
        'labels': np.array(all_labels),
        'probs':  np.array(all_probs),
    }


def run_training(model, train_loader, val_loader, criterion, optimizer,
                 scheduler, binary=True, tag=''):
    """
    Full training loop with early stopping on validation F1.
    Keeps best model weights in memory (no disk I/O during training).
    Returns history dict.
    """
    best_val_f1 = 0.0
    no_improve  = 0
    best_state  = None
    avg_metric  = 'binary' if binary else 'macro'
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss  = train_epoch(model, train_loader, criterion, optimizer)
        val_res  = evaluate(model, val_loader, criterion)

        acc = accuracy_score(val_res['labels'], val_res['preds'])
        _, _, f1, _ = precision_recall_fscore_support(
            val_res['labels'], val_res['preds'],
            average=avg_metric, zero_division=0)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_res['loss'])
        history['val_acc'].append(acc)
        history['val_f1'].append(f1)

        scheduler.step(val_res['loss'])

        print(f"[{tag}] Ep {epoch:2d}/{MAX_EPOCHS} | "
              f"Train Loss: {tr_loss:.4f} | Val Loss: {val_res['loss']:.4f} | "
              f"Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

        if f1 > best_val_f1:
            best_val_f1 = f1
            best_state  = copy.deepcopy(model.state_dict())
            no_improve  = 0
            print(f"   *** Best Val F1 = {best_val_f1:.4f}  (model checkpoint saved in memory) ***")
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"   Early stopping after epoch {epoch} (no F1 gain for {PATIENCE} epochs).")
            break

    model.load_state_dict(best_state)
    print(f"Training complete. Best Val F1 = {best_val_f1:.4f}")
    return history


print("Helper functions defined.")

In [ ]:
# ── CELL 10: EXPERIMENT 1 — Kermany Pneumonia Detection (Proposed Method) ─────
print("=" * 65)
print("EXPERIMENT 1: Kermany — Layer Freezing + Label Smoothing")
print("  Same task as A2 (binary: NORMAL vs PNEUMONIA)")
print("  Improvement: only last 2 ViT blocks + head are updated")
print("=" * 65)

model_k = build_vit(num_labels=2, freeze_n=FREEZE_N)

# CrossEntropyLoss with class weights (imbalance) + label smoothing (NEW)
criterion_k = nn.CrossEntropyLoss(weight=cw_k, label_smoothing=LABEL_SMOOTH)

# Optimise only trainable parameters (frozen params not wasted on)
optimizer_k = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_k.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler_k = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_k, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE
)

history_k = run_training(
    model_k, train_loader_k, val_loader_k,
    criterion_k, optimizer_k, scheduler_k,
    binary=True, tag='Kermany'
)

torch.save(model_k.state_dict(), SAVE_DIR + 'best_vit_kermany_a3.pth')
print("\nModel saved to Drive.")

In [ ]:
# ── CELL 11: Experiment 1 — Test Evaluation & Visualisations ─────────────────

test_res_k = evaluate(model_k, test_loader_k, criterion_k)
lbl_k  = test_res_k['labels']
pred_k = test_res_k['preds']
prob_k = test_res_k['probs']

acc_k  = accuracy_score(lbl_k, pred_k)
prec_k, rec_k, f1_k, _ = precision_recall_fscore_support(
    lbl_k, pred_k, average='binary', zero_division=0)
# Specificity = recall of NORMAL class (label 0)
_, spec_k, _, _ = precision_recall_fscore_support(
    lbl_k, pred_k, average='binary', pos_label=0, zero_division=0)
auc_k = roc_auc_score(lbl_k, prob_k[:, 1])

print("\n" + "=" * 65)
print("EXPERIMENT 1 — Test Set Results")
print("=" * 65)
print(f"  Accuracy    : {acc_k*100:.2f}%   (A2 baseline: {A2_RESULTS['Accuracy (%)']}%)")
print(f"  Recall      : {rec_k*100:.2f}%   (A2 baseline: {A2_RESULTS['Recall (%)']}%)")
print(f"  Specificity : {spec_k*100:.2f}%   (A2 baseline: {A2_RESULTS['Specificity (%)']}%)")
print(f"  F1-Score    : {f1_k*100:.2f}%   (A2 baseline: {A2_RESULTS['F1 (%)']}%)")
print(f"  AUC-ROC     : {auc_k:.4f}   (A2 baseline: {A2_RESULTS['AUC']})")
print()
print(classification_report(lbl_k, pred_k, target_names=CLASS_NAMES_K))

# ── Training Curves ─────────────────────────────────────────────────────────
eps = range(1, len(history_k['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Exp 1 — Kermany Training Curves (A3: Layer Freezing + Label Smoothing)',
             fontweight='bold', fontsize=12)

axes[0].plot(eps, history_k['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(eps, history_k['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(eps, history_k['val_acc'], 'g-o', ms=4)
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy'); axes[1].grid(True, alpha=0.4)

best_ep_k = int(np.argmax(history_k['val_f1'])) + 1
axes[2].plot(eps, history_k['val_f1'], 'm-o', ms=4)
axes[2].axvline(best_ep_k, color='red', linestyle='--', alpha=0.6, label=f'Best epoch {best_ep_k}')
axes[2].set_title('Validation F1 (Pneumonia)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'kermany_curves_a3.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Confusion Matrix ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(lbl_k, pred_k),
    display_labels=CLASS_NAMES_K
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Kermany Test Set (A3)', fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR + 'kermany_cm_a3.png', dpi=200, bbox_inches='tight')
plt.show()

# ── ROC Curve ───────────────────────────────────────────────────────────────
fpr_k, tpr_k, _ = roc_curve(lbl_k, prob_k[:, 1])
plt.figure(figsize=(5, 4))
plt.plot(fpr_k, tpr_k, 'b-', lw=2, label=f'A3 ViT Partial Fine-tune (AUC={auc_k:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Kermany (A3)', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(SAVE_DIR + 'kermany_roc_a3.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"Kermany figures saved to {SAVE_DIR}")

In [ ]:
# ── CELL 12: EXPERIMENT 2 — BloodMNIST Cross-Domain Evaluation ────────────────
print("=" * 65)
print("EXPERIMENT 2: BloodMNIST — Cross-Domain Evaluation")
print("  8-class blood cell type classification")
print("  Same ViT architecture + same improvements (freeze + label smooth)")
print("=" * 65)

model_b = build_vit(num_labels=NUM_BLOOD_CLASSES, freeze_n=FREEZE_N)

# No class weighting needed — BloodMNIST is approximately balanced
criterion_b = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

optimizer_b = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE
)

history_b = run_training(
    model_b, train_loader_b, val_loader_b,
    criterion_b, optimizer_b, scheduler_b,
    binary=False, tag='BloodMNIST'
)

torch.save(model_b.state_dict(), SAVE_DIR + 'best_vit_bloodmnist_a3.pth')
print("\nBloodMNIST model saved to Drive.")

In [ ]:
# ── CELL 13: Experiment 2 — Test Evaluation & Visualisations ─────────────────

test_res_b = evaluate(model_b, test_loader_b, criterion_b)
lbl_b  = test_res_b['labels']
pred_b = test_res_b['preds']
prob_b = test_res_b['probs']

acc_b  = accuracy_score(lbl_b, pred_b)
prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(
    lbl_b, pred_b, average='macro', zero_division=0)
auc_b = roc_auc_score(lbl_b, prob_b, multi_class='ovr', average='macro')

print("\n" + "=" * 65)
print("EXPERIMENT 2 — Test Set Results (BloodMNIST)")
print("=" * 65)
print(f"  Accuracy  (macro): {acc_b*100:.2f}%")
print(f"  Precision (macro): {prec_b*100:.2f}%")
print(f"  Recall    (macro): {rec_b*100:.2f}%")
print(f"  F1-Score  (macro): {f1_b*100:.2f}%")
print(f"  AUC-ROC (OvR macro): {auc_b:.4f}")
print()
print(classification_report(lbl_b, pred_b, target_names=CLASS_NAMES_B))

# ── Training Curves ─────────────────────────────────────────────────────────
eps_b = range(1, len(history_b['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Exp 2 — BloodMNIST Training Curves (A3: Layer Freezing + Label Smoothing)',
             fontweight='bold', fontsize=12)

axes[0].plot(eps_b, history_b['train_loss'], 'b-o', ms=4, label='Train')
axes[0].plot(eps_b, history_b['val_loss'],   'r-o', ms=4, label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(eps_b, history_b['val_acc'], 'g-o', ms=4)
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy'); axes[1].grid(True, alpha=0.4)

best_ep_b = int(np.argmax(history_b['val_f1'])) + 1
axes[2].plot(eps_b, history_b['val_f1'], 'm-o', ms=4)
axes[2].axvline(best_ep_b, color='red', linestyle='--', alpha=0.6, label=f'Best epoch {best_ep_b}')
axes[2].set_title('Validation F1 (Macro)'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'bloodmnist_curves_a3.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Confusion Matrix ────────────────────────────────────────────────────────
cm_b = confusion_matrix(lbl_b, pred_b)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm_b, display_labels=CLASS_NAMES_B).plot(
    ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — BloodMNIST Test Set (A3)', fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig(SAVE_DIR + 'bloodmnist_cm_a3.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"BloodMNIST figures saved to {SAVE_DIR}")

In [ ]:
# ── CELL 14: Final Comparison Table ──────────────────────────────────────────

print("\n" + "=" * 80)
print("FINAL RESULTS COMPARISON")
print("=" * 80)

comp = pd.DataFrame({
    'Metric': ['Accuracy (%)', 'Recall (%)', 'Specificity (%)', 'F1 (%)', 'AUC-ROC'],
    'Singh et al. 2024 (Paper)': ['97.61', '95.00', '98.00', 'N/A',   '0.96'],
    'A2: Full Fine-tune\n(Kermany, all 86M params)': [
        str(A2_RESULTS['Accuracy (%)']),
        str(A2_RESULTS['Recall (%)']),
        str(A2_RESULTS['Specificity (%)']),
        str(A2_RESULTS['F1 (%)']),
        str(A2_RESULTS['AUC']),
    ],
    'A3: Partial Fine-tune\n(Kermany, ~14M params, label smooth)': [
        f"{acc_k*100:.2f}",
        f"{rec_k*100:.2f}",
        f"{spec_k*100:.2f}",
        f"{f1_k*100:.2f}",
        f"{auc_k:.4f}",
    ],
    'A3: Cross-Domain\n(BloodMNIST, 8-class)': [
        f"{acc_b*100:.2f}",
        f"{rec_b*100:.2f}  (macro)",
        'N/A',
        f"{f1_b*100:.2f}  (macro)",
        f"{auc_b:.4f}  (OvR)",
    ],
})

pd.set_option('display.max_colwidth', 45)
pd.set_option('display.width', 200)
print(comp.to_string(index=False))
comp.to_csv(SAVE_DIR + 'final_comparison_a3.csv', index=False)
print(f"\nTable saved to {SAVE_DIR}final_comparison_a3.csv")

In [ ]:
# ── CELL 15: Bootstrap Confidence Intervals (Statistical Analysis) ────────────
#
# We use non-parametric bootstrapping (1000 resamples of the test set) to
# compute 95% confidence intervals for Kermany test metrics.
# This is appropriate because we have a single test set and cannot repeat the
# experiment independently.

N_BOOT = 1000
rng_bs = np.random.default_rng(SEED + 99)

bs_acc, bs_f1, bs_rec, bs_spec, bs_auc = [], [], [], [], []

for _ in range(N_BOOT):
    idx      = rng_bs.choice(len(lbl_k), len(lbl_k), replace=True)
    bs_lbl   = lbl_k[idx]
    bs_pred  = pred_k[idx]
    bs_prob  = prob_k[idx, 1]

    bs_acc.append(accuracy_score(bs_lbl, bs_pred))

    _, r, f, _ = precision_recall_fscore_support(
        bs_lbl, bs_pred, average='binary', zero_division=0)
    bs_f1.append(f)
    bs_rec.append(r)

    _, s, _, _ = precision_recall_fscore_support(
        bs_lbl, bs_pred, average='binary', pos_label=0, zero_division=0)
    bs_spec.append(s)

    if len(np.unique(bs_lbl)) > 1:   # skip if all same class in resample
        bs_auc.append(roc_auc_score(bs_lbl, bs_prob))

def ci_str(values, scale=100, fmt=".2f"):
    lo = np.percentile(values, 2.5)  * scale
    hi = np.percentile(values, 97.5) * scale
    mn = np.mean(values)             * scale
    return f"{mn:{fmt}}  (95% CI: [{lo:{fmt}}, {hi:{fmt}}])"

print("\n" + "-" * 65)
print("Bootstrap 95% CIs — Kermany A3 Test Set (n_boot=1000)")
print("-" * 65)
print(f"  Accuracy    : {ci_str(bs_acc)}%")
print(f"  Recall      : {ci_str(bs_rec)}%")
print(f"  Specificity : {ci_str(bs_spec)}%")
print(f"  F1-Score    : {ci_str(bs_f1)}%")
print(f"  AUC-ROC     : {ci_str(bs_auc, scale=1, fmt='.4f')}")

In [ ]:
# ── CELL 16: Final Summary ────────────────────────────────────────────────────
print("=" * 70)
print("ASSIGNMENT 3 — FINAL SUMMARY")
print("=" * 70)
print()
print("HYPOTHESIS:")
print("  Partial fine-tuning (freeze early ViT blocks) + label smoothing")
print("  produces more stable training vs A2 full fine-tune, achieves")
print("  comparable or better generalisation on Kermany, and transfers")
print("  effectively to the cross-domain BloodMNIST benchmark.")
print()
print("IMPROVEMENTS APPLIED:")
print(f"  1. Layer Freezing  : blocks 0-{FREEZE_N-1} frozen (~{FREEZE_N}/12 encoder blocks)")
print(f"     Trainable params : ~14M vs ~86M in A2 (84% reduction)")
print(f"     Effect           : prevents training collapse (0 loss / F1=1)")
print(f"  2. Label Smoothing : ε = {LABEL_SMOOTH}")
print(f"     Effect           : regularises loss, improves calibration")
print(f"  3. Val Transform Fix: val_ds now uses eval_tfm (no augmentation)")
print(f"     Effect           : cleaner val signal, correct early stopping")
print()
print("EXPERIMENT 1 (Kermany — same task as A2):")
print(f"  Test Accuracy   : {acc_k*100:.2f}%   [A2: {A2_RESULTS['Accuracy (%)']}%]")
print(f"  Test Recall     : {rec_k*100:.2f}%   [A2: {A2_RESULTS['Recall (%)']}%]")
print(f"  Test Specificity: {spec_k*100:.2f}%   [A2: {A2_RESULTS['Specificity (%)']}%]")
print(f"  Test F1         : {f1_k*100:.2f}%   [A2: {A2_RESULTS['F1 (%)']}%]")
print(f"  Test AUC-ROC    : {auc_k:.4f}    [A2: {A2_RESULTS['AUC']}]")
print()
print("EXPERIMENT 2 (BloodMNIST — cross-domain, 8 classes):")
print(f"  Test Accuracy   : {acc_b*100:.2f}%  (macro)")
print(f"  Test F1-Score   : {f1_b*100:.2f}%  (macro)")
print(f"  Test AUC-ROC    : {auc_b:.4f}   (OvR macro)")
print()
print(f"All outputs saved to: {SAVE_DIR}")
print("=" * 70)